In [ ]:
import sys, os
REPO_DIR = os.path.dirname(os.path.abspath("__file__"))
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

In [ ]:
import matplotlib.pyplot as plt
import warnings
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
import pandas as pd
from utils import  provide_x_z, train_xgb_simple, convert_scores_to_ranks, preprocessing_data
from sklearn.model_selection import KFold
from crt_cgan import ConditionalGAN, crt_calibration_efficient
import numpy as np
from tqdm.notebook import tqdm

In [ ]:
from methods import (condor_score, nkci_score, nhsic_score, kcondor_score, cmi_score, 
                     hsic_hyppo_score, partial_dcorr_score, pdnhsic_v2, 
                     Kcondor_v2, Kcondor_v2_opt, Kcondor_v2_opt2, Kcondor_v2_precomputed_factory,
                     Kcondor_v3, kci_pval, nhsic, partial_corr_pg_score)

In [ ]:
methods ={
    "Condor": Kcondor_v2,#Kcondor_v2_precomputed_factory, #,
    "nKCI": nkci_score,
    'nhsic': nhsic,
    'cmi_score': cmi_score, 
    'partial_dcorr_score': partial_corr_pg_score#partial_dcorr_score
    ##'kci_pval': kci_pval,
    }

In [ ]:
K = 5
linspace_dim = 7
betas = np.linspace(0., 1, linspace_dim)
B = 200
#features_Z = [['African_American'],['Female'],['random']]
dataset_name = "adults"#"propublica"  # 
s_size = 500

In [ ]:
DATA_DIR = os.path.join(REPO_DIR, "data")

if dataset_name == "adults":
    data_path = os.path.join(DATA_DIR, "adult.csv")
    y_name = 'income'
    features_Z = [['gender'],['race'],['random']]
elif dataset_name == "propublica":
    data_path = os.path.join(DATA_DIR, "propublica_data_for_fairml.csv")
    y_name = 'Two_yr_Recidivism'
    features_Z = [['African_American'],['Female'],['random']]
df_all = pd.read_csv(data_path)
print(df_all.shape)

In [ ]:
#for i,f_p in enumerate(features_Z):
f_p = features_Z[0]

In [ ]:
df_all.head()

In [ ]:
df_p = preprocessing_data(df_all, 
                          y_name=y_name,
                          dataset_name=dataset_name)

In [ ]:
df_p.head()

In [ ]:
# add a random column continous with values between 0 and 1
df_p['random'] = np.random.rand(df_p.shape[0])

In [ ]:
# get X, Z, y for the balanced sample
X_np, Z_np, y, X_np_all, Z_np_all, y_all = provide_x_z(df_p, y_name=y_name, f_p=f_p, sample_size_per_class=s_size)
# train a ranking model on the entire dataset
print(f"Training dei modelli di ranking per features {f_p}...")
model_o = train_xgb_simple(pd.DataFrame(X_np_all), y_all)["model"]
model_p = train_xgb_simple(pd.DataFrame(Z_np_all), y_all)["model"]
preds_o = model_o.predict_proba(pd.DataFrame(X_np))[:,1]
preds_p = model_p.predict_proba(pd.DataFrame(Z_np))[:,1]

In [ ]:
# train K cGAN generators
kf = KFold(n_splits=K, shuffle=True, random_state=42)
kf_splits = list(kf.split(X_np))
print(f"Pre-addestramento di {K} generatori cGAN per il dataset...")
trained_generators = []
for train_idx, _ in tqdm(kf_splits, desc="Training generators"):
    generator = ConditionalGAN(x_dim=X_np.shape[1], z_dim=Z_np.shape[1])
    generator.fit(X_np[train_idx], Z_np[train_idx])
    trained_generators.append(generator)
print("Addestramento completato.")


results = {name: [] for name in methods}
for beta in tqdm(betas, desc="Beta Loop"):
    R = convert_scores_to_ranks((1 - beta) * preds_o + beta * preds_p)
    #sss = (1 - beta) * preds_o + beta * preds_p
    for name, func in methods.items():
        p_val = crt_calibration_efficient(X_np, Z_np, R, scoring_function=func, kf_splits=kf_splits, trained_generators=trained_generators, B=B)
        results[name].append(p_val)

In [ ]:
results_df = pd.DataFrame(results, index=betas).rename_axis(index='Beta', columns='Method')

In [ ]:
custom_cmap = LinearSegmentedColormap.from_list(
    "pvalue_cmap",
    [(0.0, "red"), (0.05, "white"), (1.0, "green")]
    )
plt.figure(figsize=(10, 6))
ax = sns.heatmap(results_df.T, annot=True, cmap=custom_cmap, fmt=".3f", vmin=0, vmax=1, cbar_kws={'label': 'P-Value'})
ax.set_xticklabels([f"{beta:.3f}" for beta in betas])
plt.title('Heatmap P-Values: Metodo vs. Beta (cGAN, ottimizzato)')
plt.show()

In [ ]:
# save results_all to pickle
import pickle
with open(f"exp_betas_dataset_name_{dataset_name}_f_p_{f_p[0]}_B_{B}_K_{K}_betas_{linspace_dim}_size_{s_size}.pkl", "wb") as f:
    pickle.dump(results_df, f)